# ADS-B transmitter identification

All `data/*.npy` captures are combined. ICAO is used only as the label and models receive only the 80-sample ADS-B preamble. The main run keeps every eligible class; the two-class run keeps only the two largest classes before splitting.


In [1]:
from pathlib import Path

CONFIG = {
    "data_dir": Path("data"),
    "file_pattern": "*.npy",
    "known_class_count": 2,
    "min_records_per_class": 6,
    "sample_rate_hz": 10e6,
    "samples_per_half_symbol": 5,
    "preamble_half_symbols": 16,
    "message_bits": 112,
    "record_len": 1200,
    "preamble_len": 80,
    "group_gap_seconds": 5.0,
    "requested_folds": 5,
    "seed": 42,
    "batch_size": 64,
    "epochs": 15,
    "learning_rate": 1e-3,
    "weight_decay": 2e-4,
    "dropout": 0.30,
    "cnn_channels": (16, 32, 48),
    "tcn_channels": 24,
    "gru_hidden": 32,
    "random_forest_trees": 200,
    "models": (
        "physical_feature_svm",
        "physical_feature_random_forest",
        "preamble_cnn",
        "preamble_tcn",
        "preamble_gru_rnn",
    ),
}

TWO_CLASS = CONFIG["known_class_count"] == 2
RUN_NAME = "two_class" if TWO_CLASS else "all_eligible_classes"
CONFIG["output_dir"] = Path(
    "adsb_specific_transmitter_two_class_outputs"
    if TWO_CLASS else "adsb_specific_transmitter_raw_outputs"
)
CONFIG["figure_dir"] = Path("notes/figures") / f"20260921_{RUN_NAME}"
CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
CONFIG["figure_dir"].mkdir(parents=True, exist_ok=True)
CONFIG


{'data_dir': PosixPath('data'),
 'file_pattern': '*.npy',
 'known_class_count': 2,
 'min_records_per_class': 6,
 'sample_rate_hz': 10000000.0,
 'samples_per_half_symbol': 5,
 'preamble_half_symbols': 16,
 'message_bits': 112,
 'record_len': 1200,
 'preamble_len': 80,
 'group_gap_seconds': 5.0,
 'requested_folds': 5,
 'seed': 42,
 'batch_size': 64,
 'epochs': 15,
 'learning_rate': 0.001,
 'weight_decay': 0.0002,
 'dropout': 0.3,
 'cnn_channels': (16, 32, 48),
 'tcn_channels': 24,
 'gru_hidden': 32,
 'random_forest_trees': 200,
 'models': ('physical_feature_svm',
  'physical_feature_random_forest',
  'preamble_cnn',
  'preamble_tcn',
  'preamble_gru_rnn'),
 'output_dir': PosixPath('adsb_specific_transmitter_two_class_outputs'),
 'figure_dir': PosixPath('notes/figures/20260921_two_class')}

In [2]:
import hashlib
import json
import pickle
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyModeS as pms
import torch
import torch.nn as nn

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torch.utils.data import DataLoader, Dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
print("Device:", DEVICE)


Device: cpu


## Data


In [3]:
def find_capture_files(data_dir, pattern):
    for directory in [data_dir, Path("../") / data_dir, Path("../../") / data_dir]:
        files = sorted(path.resolve() for path in directory.glob(pattern) if path.is_file())
        if files:
            return files
    raise FileNotFoundError(f"No {pattern} files found in {data_dir}")


def load_capture(path, first_index):
    rows = []
    with path.open("rb") as stream:
        source_index = 0
        while stream.tell() < path.stat().st_size:
            timestamp = np.load(stream, allow_pickle=True)
            iq = np.load(stream, allow_pickle=False)
            iq = np.asarray(iq)
            if iq.ndim != 1 or not np.iscomplexobj(iq):
                raise ValueError(f"Invalid I/Q array in {path.name}, record {source_index}")
            rows.append({
                "record_index": first_index + source_index,
                "source_record_index": source_index,
                "capture_file": path.name,
                "session_id": path.stem,
                "timestamp": pd.Timestamp(timestamp.item()),
                "sample_count": len(iq),
                "iq": iq.astype(np.complex64, copy=False),
            })
            source_index += 1
    return pd.DataFrame(rows)


files = find_capture_files(CONFIG["data_dir"], CONFIG["file_pattern"])
frames = []
next_index = 0
for path in files:
    frame = load_capture(path, next_index)
    frames.append(frame)
    next_index += len(frame)
capture_records = pd.concat(frames, ignore_index=True)


def demodulate(iq):
    if len(iq) != CONFIG["record_len"]:
        raise ValueError("Unexpected record length")
    halves = np.abs(iq).reshape(-1, CONFIG["samples_per_half_symbol"]).mean(axis=1)
    payload = halves[CONFIG["preamble_half_symbols"]:]
    bits = payload[0::2] > payload[1::2]
    packed = np.packbits(bits).tobytes()
    return packed.hex().upper(), hashlib.sha256(packed).hexdigest()


decoded = []
for row in capture_records.itertuples(index=False):
    result = {"record_index": row.record_index, "icao": None, "payload_hash": None, "decode_ok": False}
    try:
        message, payload_hash = demodulate(row.iq)
        if int(pms.crc(message)) == 0:
            result.update(icao=str(pms.adsb.icao(message)).upper(), payload_hash=payload_hash, decode_ok=True)
    except Exception:
        pass
    decoded.append(result)

records = capture_records.merge(pd.DataFrame(decoded), on="record_index", validate="one_to_one")
valid_records = records[records["decode_ok"]].copy()

capture_file_audit = valid_records.groupby(["capture_file", "session_id"], as_index=False).agg(
    records=("record_index", "size"),
    capture_start=("timestamp", "min"),
    capture_end=("timestamp", "max"),
)
display(capture_file_audit)
print(f"CRC-valid: {len(valid_records):,}/{len(records):,}")
print("ICAO classes:", valid_records["icao"].nunique())


,capture_file,session_id,records,capture_start,capture_end
0,050926.npy,050926,558,2026-08-25 16:30:54.108478,2026-08-25 18:32:37.533011
1,150926.npy,150926,998,2026-08-25 20:32:26.372398,2026-08-26 01:40:45.997423


CRC-valid: 1,556/1,556
ICAO classes: 62


## Preamble input


In [4]:
PULSE_HALVES = np.zeros(CONFIG["preamble_half_symbols"], dtype=bool)
ACTIVE_HALF_INDICES = np.array([0, 2, 7, 9])
PULSE_HALVES[ACTIVE_HALF_INDICES] = True
ACTIVE_MASK = np.repeat(PULSE_HALVES, CONFIG["samples_per_half_symbol"])


def moment_features(values):
    values = np.asarray(values, dtype=np.float64)
    mean = values.mean()
    std = values.std() + 1e-12
    z = (values - mean) / std
    return mean, std, np.mean(z ** 3), np.mean(z ** 4)


def preamble_features(preamble):
    active = preamble[ACTIVE_MASK]
    quiet = preamble[~ACTIVE_MASK]
    pulses = np.stack([
        preamble[i * CONFIG["samples_per_half_symbol"]:(i + 1) * CONFIG["samples_per_half_symbol"]]
        for i in ACTIVE_HALF_INDICES
    ])
    amplitude = np.abs(active)
    values = list(moment_features(amplitude))
    phase_steps = np.angle(pulses[:, 1:] * np.conj(pulses[:, :-1])).ravel()
    mean_step = np.angle(np.mean(np.exp(1j * phase_steps)))
    centred_steps = np.angle(np.exp(1j * (phase_steps - mean_step)))
    i_power = np.mean(active.real ** 2) + 1e-12
    q_power = np.mean(active.imag ** 2) + 1e-12
    correlation = np.corrcoef(active.real, active.imag)[0, 1]
    correlation = correlation if np.isfinite(correlation) else 0.0
    prototype = pulses.mean(axis=0)
    pulse_power = np.mean(np.abs(pulses) ** 2, axis=1)
    values += [
        amplitude.max() / (np.sqrt(np.mean(amplitude ** 2)) + 1e-12),
        np.mean(amplitude ** 2),
        np.mean(np.abs(quiet) ** 2),
        10 * np.log10((np.mean(amplitude ** 2) + 1e-12) / (np.mean(np.abs(quiet) ** 2) + 1e-12)),
        mean_step * CONFIG["sample_rate_hz"] / (2 * np.pi),
        np.std(centred_steps),
        10 * np.log10(i_power / q_power),
        correlation,
        np.std(pulse_power) / (np.mean(pulse_power) + 1e-12),
    ]
    values += np.abs(pulses).mean(axis=0).tolist()
    values += prototype.real.tolist()
    values += prototype.imag.tolist()
    values += np.angle(prototype).tolist()
    return np.asarray(values, dtype=np.float32)


FEATURE_NAMES = [
    "amplitude_mean", "amplitude_std", "amplitude_skew", "amplitude_kurtosis",
    "crest_factor", "active_power", "quiet_power", "preamble_snr_db",
    "carrier_offset_hz", "phase_step_std", "iq_power_imbalance_db",
    "iq_correlation", "pulse_power_variation",
]
FEATURE_NAMES += [f"pulse_amplitude_{i}" for i in range(5)]
FEATURE_NAMES += [f"pulse_i_{i}" for i in range(5)]
FEATURE_NAMES += [f"pulse_q_{i}" for i in range(5)]
FEATURE_NAMES += [f"pulse_phase_{i}" for i in range(5)]

counts = valid_records["icao"].value_counts()
eligible = counts[counts >= CONFIG["min_records_per_class"]]
class_names = eligible.index.tolist()
if CONFIG["known_class_count"] is not None:
    class_names = class_names[:CONFIG["known_class_count"]]

selected = valid_records[valid_records["icao"].isin(class_names)].copy().reset_index(drop=True)
class_to_index = {name: index for index, name in enumerate(class_names)}
y = selected["icao"].map(class_to_index).to_numpy()

raw_preambles = np.stack([
    np.asarray(iq[:CONFIG["preamble_len"]], dtype=np.complex64) for iq in selected["iq"]
])
X_preamble = np.stack([raw_preambles.real, raw_preambles.imag], axis=1).astype(np.float32)
X_physical = np.stack([preamble_features(values) for values in raw_preambles])
model_metadata = selected[[
    "record_index", "source_record_index", "capture_file", "session_id",
    "timestamp", "icao", "payload_hash",
]].copy()

source_iq = records.set_index("record_index")["iq"]
check = np.stack([source_iq.loc[index][:CONFIG["preamble_len"]] for index in model_metadata["record_index"]])
assert np.array_equal(raw_preambles, check)
assert X_preamble.shape[1:] == (2, 80)
assert X_physical.shape[1] == len(FEATURE_NAMES) == 33

class_session_counts = pd.crosstab(model_metadata["icao"], model_metadata["capture_file"]).reindex(class_names)
display(pd.DataFrame({
    "records": model_metadata["icao"].value_counts().reindex(class_names),
    "sessions": model_metadata.groupby("icao")["session_id"].nunique().reindex(class_names),
}))

axis = class_session_counts.plot(kind="bar", stacked=True, figsize=(12, 4))
axis.set(title="Records per retained class", xlabel="ICAO class", ylabel="Records")
plt.tight_layout()
plt.savefig(CONFIG["figure_dir"] / "class_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

example = 0
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(X_preamble[example, 0], label="I")
axes[0].plot(X_preamble[example, 1], label="Q")
axes[0].set_ylabel("Raw I/Q")
axes[0].legend()
axes[1].plot(np.abs(raw_preambles[example]), label="Magnitude")
for start in ACTIVE_HALF_INDICES * CONFIG["samples_per_half_symbol"]:
    axes[1].axvspan(start, start + CONFIG["samples_per_half_symbol"], color="0.9")
axes[1].set(xlabel="Preamble sample", ylabel="Magnitude")
axes[1].legend()
fig.suptitle(f"Example preamble: {model_metadata.loc[example, 'icao']}")
plt.tight_layout()
plt.savefig(CONFIG["figure_dir"] / "preamble_input.png", dpi=180, bbox_inches="tight")
plt.show()

print("Preamble tensor:", X_preamble.shape)
print("Physical features:", X_physical.shape)


,records,sessions
icao,,
7C0B96,246,1
7C802A,64,1


Preamble tensor: (310, 2, 80)
Physical features: (310, 33)


/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_7121/1702643194.py:98: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_7121/1702643194.py:114: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Grouped folds


In [5]:
def connected_groups(metadata):
    output = np.empty(len(metadata), dtype=object)
    for icao, frame in metadata.groupby("icao", sort=True):
        positions = frame.index.to_numpy()
        local = {position: i for i, position in enumerate(positions)}
        parent = list(range(len(positions)))

        def find(i):
            while parent[i] != i:
                parent[i] = parent[parent[i]]
                i = parent[i]
            return i

        def union(a, b):
            a, b = find(a), find(b)
            if a != b:
                parent[b] = a

        for _, same_payload in frame.groupby("payload_hash"):
            members = same_payload.index.tolist()
            for member in members[1:]:
                union(local[members[0]], local[member])

        for _, same_session in frame.groupby("session_id"):
            ordered = same_session.sort_values("timestamp").index.tolist()
            for left, right in zip(ordered, ordered[1:]):
                gap = (metadata.loc[right, "timestamp"] - metadata.loc[left, "timestamp"]).total_seconds()
                if gap <= CONFIG["group_gap_seconds"]:
                    union(local[left], local[right])

        roots = {}
        for position in positions:
            root = find(local[position])
            roots.setdefault(root, len(roots))
            output[position] = f"{icao}:group_{roots[root]:03d}"
    return output


groups = connected_groups(model_metadata)
model_metadata["group_id"] = groups
groups_per_class = model_metadata.groupby("icao")["group_id"].nunique().reindex(class_names)
n_splits = min(CONFIG["requested_folds"], int(groups_per_class.min()))
if n_splits < 2:
    raise ValueError("At least two groups are needed for every class")


def make_folds(labels, group_ids, n_folds, seed):
    rng = np.random.default_rng(seed)
    fold_groups = [set() for _ in range(n_folds)]
    total_load = np.zeros(n_folds, dtype=int)

    for class_id in np.unique(labels):
        class_groups, sizes = np.unique(group_ids[labels == class_id], return_counts=True)
        order = rng.permutation(len(class_groups))
        class_groups, sizes = class_groups[order], sizes[order]
        order = np.argsort(-sizes, kind="stable")
        class_load = np.zeros(n_folds, dtype=int)
        for group_id, size in zip(class_groups[order], sizes[order]):
            choices = np.flatnonzero(class_load == class_load.min())
            fold = int(choices[np.argmin(total_load[choices])])
            fold_groups[fold].add(group_id)
            class_load[fold] += int(size)
            total_load[fold] += int(size)

    folds = []
    for validation_groups in fold_groups:
        validation = np.flatnonzero(np.isin(group_ids, list(validation_groups)))
        training = np.flatnonzero(~np.isin(group_ids, list(validation_groups)))
        folds.append((training, validation))
    return folds


folds = make_folds(y, groups, n_splits, CONFIG["seed"])
fold_rows = []
for fold, (train_index, val_index) in enumerate(folds, 1):
    train_groups = set(groups[train_index])
    val_groups = set(groups[val_index])
    train_hashes = set(model_metadata.loc[train_index, "payload_hash"])
    val_hashes = set(model_metadata.loc[val_index, "payload_hash"])
    assert train_groups.isdisjoint(val_groups)
    assert train_hashes.isdisjoint(val_hashes)
    assert set(y[train_index]) == set(range(len(class_names)))
    assert set(y[val_index]) == set(range(len(class_names)))
    fold_rows.append({
        "fold": fold,
        "training_records": len(train_index),
        "validation_records": len(val_index),
        "classes": len(class_names),
        "group_overlap": len(train_groups & val_groups),
        "payload_overlap": len(train_hashes & val_hashes),
    })

fold_audit = pd.DataFrame(fold_rows)
display(fold_audit)
fold_audit.set_index("fold")[["training_records", "validation_records"]].plot(
    kind="bar", figsize=(8, 4)
)
plt.ylabel("Records")
plt.title("Grouped fold sizes")
plt.tight_layout()
plt.savefig(CONFIG["figure_dir"] / "fold_sizes.png", dpi=180, bbox_inches="tight")
plt.show()
print(f"Using {n_splits} folds; groups per class range {groups_per_class.min()}--{groups_per_class.max()}")


,fold,training_records,validation_records,classes,group_overlap,payload_overlap
0,1,119,191,2,0,0
1,2,287,23,2,0,0
2,3,291,19,2,0,0
3,4,292,18,2,0,0
4,5,251,59,2,0,0


Using 5 folds; groups per class range 7--31


/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_7121/2140617103.py:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Models

The SVM and random forest use 33 preamble features. The CNN, TCN and GRU use the $2\times80$ I/Q sequence.


In [6]:
class ArrayDataset(Dataset):
    def __init__(self, values, labels=None):
        self.values = torch.as_tensor(values, dtype=torch.float32)
        self.labels = None if labels is None else torch.as_tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.values)

    def __getitem__(self, index):
        if self.labels is None:
            return self.values[index]
        return self.values[index], self.labels[index]


class PreambleCNN(nn.Module):
    def __init__(self, classes):
        super().__init__()
        c1, c2, c3 = CONFIG["cnn_channels"]
        self.network = nn.Sequential(
            nn.Conv1d(2, c1, 7, padding=3), nn.BatchNorm1d(c1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(c1, c2, 5, padding=2), nn.BatchNorm1d(c2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(c2, c3, 3, padding=1), nn.BatchNorm1d(c3), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(c3, classes),
        )

    def forward(self, values):
        return self.network(values)


class TCNBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm1d(channels), nn.ReLU(), nn.Dropout(CONFIG["dropout"] / 2),
            nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm1d(channels),
        )

    def forward(self, values):
        return torch.relu(values + self.block(values))


class PreambleTCN(nn.Module):
    def __init__(self, classes):
        super().__init__()
        channels = CONFIG["tcn_channels"]
        self.input_layer = nn.Conv1d(2, channels, 1)
        self.blocks = nn.Sequential(*(TCNBlock(channels, d) for d in (1, 2, 4)))
        self.output_layer = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(CONFIG["dropout"]),
            nn.Linear(channels, classes),
        )

    def forward(self, values):
        return self.output_layer(self.blocks(self.input_layer(values)))


class PreambleGRU(nn.Module):
    def __init__(self, classes):
        super().__init__()
        self.gru = nn.GRU(2, CONFIG["gru_hidden"], batch_first=True, bidirectional=True)
        self.output_layer = nn.Sequential(
            nn.LayerNorm(2 * CONFIG["gru_hidden"]),
            nn.Dropout(CONFIG["dropout"]),
            nn.Linear(2 * CONFIG["gru_hidden"], classes),
        )

    def forward(self, values):
        _, hidden = self.gru(values.transpose(1, 2))
        return self.output_layer(torch.cat([hidden[-2], hidden[-1]], dim=1))


MODEL_NAMES = {
    "physical_feature_svm": "Physical-feature RBF SVM",
    "physical_feature_random_forest": "Physical-feature random forest",
    "preamble_cnn": "Compact 1D CNN",
    "preamble_tcn": "Compact TCN",
    "preamble_gru_rnn": "Compact bidirectional GRU",
}

sample_models = {
    "preamble_cnn": PreambleCNN(len(class_names)),
    "preamble_tcn": PreambleTCN(len(class_names)),
    "preamble_gru_rnn": PreambleGRU(len(class_names)),
}
neural_parameters = {
    name: sum(parameter.numel() for parameter in model.parameters())
    for name, model in sample_models.items()
}
display(pd.DataFrame([
    {"model": MODEL_NAMES[name], "trainable_parameters": count}
    for name, count in neural_parameters.items()
]))
del sample_models


,model,trainable_parameters
0,Compact 1D CNN,7778
1,Compact TCN,10922
2,Compact bidirectional GRU,7170


In [7]:
def loader(values, labels=None, shuffle=False):
    return DataLoader(
        ArrayDataset(values, labels), batch_size=CONFIG["batch_size"],
        shuffle=shuffle, num_workers=0,
    )


def train_neural(model, train_values, train_labels, seed):
    set_seed(seed)
    model = model.to(DEVICE)
    counts = np.bincount(train_labels, minlength=len(class_names)).astype(np.float32)
    weights = counts.sum() / (len(class_names) * np.maximum(counts, 1))
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, device=DEVICE))
    optimiser = torch.optim.AdamW(
        model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"]
    )
    history = []
    for _ in range(CONFIG["epochs"]):
        model.train()
        loss_sum = 0.0
        for batch, targets in loader(train_values, train_labels, shuffle=True):
            batch, targets = batch.to(DEVICE), targets.to(DEVICE)
            optimiser.zero_grad()
            loss = criterion(model(batch), targets)
            loss.backward()
            optimiser.step()
            loss_sum += loss.item() * len(targets)
        history.append(loss_sum / len(train_labels))
    return model, history


@torch.no_grad()
def predict_neural(model, values):
    model.eval()
    return np.concatenate([
        torch.argmax(model(batch.to(DEVICE)), dim=1).cpu().numpy()
        for batch in loader(values)
    ])


def scores(targets, predictions):
    return {
        "accuracy": accuracy_score(targets, predictions),
        "balanced_accuracy": balanced_accuracy_score(targets, predictions),
        "macro_f1": f1_score(targets, predictions, average="macro", zero_division=0),
    }


## Training and results


In [8]:
fold_results = []
prediction_rows = []
complexity_rows = []
feature_importance_rows = []
training_histories = {}

for fold_number, (train_index, val_index) in enumerate(folds, 1):
    print(f"Fold {fold_number}/{n_splits}")
    channel_mean = X_preamble[train_index].mean(axis=(0, 2), keepdims=True)
    channel_std = X_preamble[train_index].std(axis=(0, 2), keepdims=True) + 1e-6
    scaled_train = (X_preamble[train_index] - channel_mean) / channel_std
    scaled_validation = (X_preamble[val_index] - channel_mean) / channel_std

    for model_number, experiment in enumerate(CONFIG["models"]):
        seed = CONFIG["seed"] + 100 * fold_number + model_number

        if experiment == "physical_feature_svm":
            model = make_pipeline(
                StandardScaler(),
                SVC(C=3.0, kernel="rbf", gamma="scale", class_weight="balanced"),
            )
            model.fit(X_physical[train_index], y[train_index])
            predictions = model.predict(X_physical[val_index])
            support_vectors = model.named_steps["svc"].support_vectors_.shape[0]
            complexity_rows.append({
                "experiment": experiment, "fold": fold_number,
                "parameters": np.nan, "support_vectors": support_vectors,
                "tree_nodes": np.nan, "fitted_size_mb": len(pickle.dumps(model)) / 1e6,
            })

        elif experiment == "physical_feature_random_forest":
            model = RandomForestClassifier(
                n_estimators=CONFIG["random_forest_trees"], min_samples_leaf=2,
                max_features="sqrt", class_weight="balanced_subsample",
                n_jobs=-1, random_state=seed,
            )
            model.fit(X_physical[train_index], y[train_index])
            predictions = model.predict(X_physical[val_index])
            for name, importance in zip(FEATURE_NAMES, model.feature_importances_):
                feature_importance_rows.append({
                    "fold": fold_number, "feature": name, "importance": importance,
                })
            complexity_rows.append({
                "experiment": experiment, "fold": fold_number,
                "parameters": np.nan, "support_vectors": np.nan,
                "tree_nodes": sum(tree.tree_.node_count for tree in model.estimators_),
                "fitted_size_mb": len(pickle.dumps(model)) / 1e6,
            })

        else:
            if experiment == "preamble_cnn":
                model = PreambleCNN(len(class_names))
            elif experiment == "preamble_tcn":
                model = PreambleTCN(len(class_names))
            else:
                model = PreambleGRU(len(class_names))
            model, history = train_neural(model, scaled_train, y[train_index], seed)
            predictions = predict_neural(model, scaled_validation)
            parameter_count = sum(parameter.numel() for parameter in model.parameters())
            complexity_rows.append({
                "experiment": experiment, "fold": fold_number,
                "parameters": parameter_count, "support_vectors": np.nan,
                "tree_nodes": np.nan, "fitted_size_mb": parameter_count * 4 / 1e6,
            })
            training_histories[(experiment, fold_number)] = history

        metrics = scores(y[val_index], predictions)
        fold_results.append({
            "experiment": experiment, "fold": fold_number,
            "n_train": len(train_index), "n_validation": len(val_index), **metrics,
        })
        for position, target, prediction in zip(val_index, y[val_index], predictions):
            prediction_rows.append({
                "experiment": experiment, "fold": fold_number,
                "record_index": int(model_metadata.loc[position, "record_index"]),
                "capture_file": model_metadata.loc[position, "capture_file"],
                "true_class": class_names[int(target)],
                "predicted_class": class_names[int(prediction)],
            })
        print(f"  {MODEL_NAMES[experiment]:32s} balanced={metrics['balanced_accuracy']:.3f}")

results_df = pd.DataFrame(fold_results)
predictions_df = pd.DataFrame(prediction_rows)
complexity_df = pd.DataFrame(complexity_rows)
feature_importance_df = pd.DataFrame(feature_importance_rows)

overall_rows = []
for experiment in CONFIG["models"]:
    subset = predictions_df[predictions_df["experiment"] == experiment]
    overall_rows.append({
        "experiment": experiment,
        "model": MODEL_NAMES[experiment],
        **scores(subset["true_class"], subset["predicted_class"]),
    })
overall_summary = pd.DataFrame(overall_rows).sort_values("balanced_accuracy", ascending=False)
display(overall_summary)


Fold 1/5
  Physical-feature RBF SVM         balanced=0.435


  Physical-feature random forest   balanced=0.363

  Compact 1D CNN                   balanced=0.374


  Compact TCN                      balanced=0.555


  Compact bidirectional GRU        balanced=0.388
Fold 2/5
  Physical-feature RBF SVM         balanced=0.779
  Physical-feature random forest   balanced=0.562


...

  Compact bidirectional GRU        balanced=0.567
Fold 5/5
  Physical-feature RBF SVM         balanced=0.544
  Physical-feature random forest   balanced=0.500


  Compact 1D CNN                   balanced=0.653


  Compact TCN                      balanced=0.633


  Compact bidirectional GRU        balanced=0.626


,experiment,model,accuracy,balanced_accuracy,macro_f1
3,preamble_tcn,Compact TCN,0.638710,0.610518,0.567212
4,preamble_gru_rnn,Compact bidirectional GRU,0.690323,0.591019,0.577872
2,preamble_cnn,Compact 1D CNN,0.690323,0.585239,0.573981
0,physical_feature_svm,Physical-feature RBF SVM,0.590323,0.447091,0.448993
1,physical_feature_random_forest,Physical-feature random forest,0.483871,0.316438,0.336757


In [9]:
neural_names = ["preamble_cnn", "preamble_tcn", "preamble_gru_rnn"]
fig, axis = plt.subplots(figsize=(8, 4))
for experiment in neural_names:
    histories = np.stack([
        training_histories[(experiment, fold)] for fold in range(1, n_splits + 1)
    ])
    axis.plot(np.arange(1, CONFIG["epochs"] + 1), histories.mean(axis=0), label=MODEL_NAMES[experiment])
axis.set(xlabel="Epoch", ylabel="Mean training loss", title="Neural model training")
axis.legend()
plt.tight_layout()
plt.savefig(CONFIG["figure_dir"] / "training_loss.png", dpi=180, bbox_inches="tight")
plt.show()

plot_data = overall_summary.sort_values("balanced_accuracy")
positions = np.arange(len(plot_data))
fig, axis = plt.subplots(figsize=(9, 5))
axis.barh(positions - 0.18, plot_data["balanced_accuracy"], 0.36, label="Balanced accuracy")
axis.barh(positions + 0.18, plot_data["macro_f1"], 0.36, label="Macro F1")
axis.axvline(1 / len(class_names), color="black", linestyle="--", label="Balanced chance")
axis.set_yticks(positions, plot_data["model"])
axis.set_xlabel("Out-of-fold score")
axis.set_title("Model comparison")
axis.legend()
plt.tight_layout()
plt.savefig(CONFIG["figure_dir"] / "model_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

matrix_columns = len(CONFIG["models"]) if len(class_names) <= 10 else 3
matrix_rows = int(np.ceil(len(CONFIG["models"]) / matrix_columns))
fig, axes = plt.subplots(
    matrix_rows, matrix_columns,
    figsize=(3.6 * matrix_columns, 4.0 * matrix_rows),
    squeeze=False, constrained_layout=True,
)
for axis, experiment in zip(axes.ravel(), CONFIG["models"]):
    subset = predictions_df[predictions_df["experiment"] == experiment]
    matrix = confusion_matrix(
        subset["true_class"], subset["predicted_class"],
        labels=class_names, normalize="true",
    )
    image = axis.imshow(matrix, vmin=0, vmax=1, aspect="auto")
    axis.set_title(MODEL_NAMES[experiment], fontsize=10)
    axis.set_xlabel("Predicted", fontsize=9)
    axis.set_ylabel("True", fontsize=9)
    if len(class_names) <= 10:
        axis.set_xticks(range(len(class_names)), class_names, rotation=45, ha="right")
        axis.set_yticks(range(len(class_names)), class_names)
        axis.tick_params(labelsize=8)
    else:
        axis.set_xticks([])
        axis.set_yticks([])
for axis in axes.ravel()[len(CONFIG["models"]):]:
    axis.axis("off")
active_axes = axes.ravel()[:len(CONFIG["models"])].tolist()
fig.colorbar(image, ax=active_axes, label="Per-class fraction", shrink=0.8, pad=0.02)
fig.suptitle("Normalised confusion matrices")
fig.savefig(CONFIG["figure_dir"] / "confusion_matrices.png", dpi=180, bbox_inches="tight")
plt.show()

importance = feature_importance_df.groupby("feature")["importance"].mean().nlargest(10).sort_values()
fig, axis = plt.subplots(figsize=(8, 5))
importance.plot(kind="barh", ax=axis)
axis.set_xlabel("Mean random-forest importance")
axis.set_title("Most used physical features")
fig.tight_layout()
fig.savefig(CONFIG["figure_dir"] / "feature_importance.png", dpi=180, bbox_inches="tight")
plt.show()


/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_7121/2634214598.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_7121/2634214598.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_7121/2634214598.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/k3/nbkql0y10tl_bv2gzn2jr_3w0000gn/T/ipykernel_7121/2634214598.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
output_dir = CONFIG["output_dir"]
results_df.to_csv(output_dir / "grouped_fold_results.csv", index=False)
overall_summary.to_csv(output_dir / "aggregate_results.csv", index=False)
predictions_df.to_csv(output_dir / "out_of_fold_predictions.csv", index=False)
fold_audit.to_csv(output_dir / "fold_leakage_audit.csv", index=False)
complexity_df.to_csv(output_dir / "model_specs.csv", index=False)
feature_importance_df.to_csv(output_dir / "random_forest_feature_importance.csv", index=False)
capture_file_audit.to_csv(output_dir / "capture_file_audit.csv", index=False)
class_session_counts.to_csv(output_dir / "class_session_counts.csv")

assignments = model_metadata[[
    "record_index", "source_record_index", "capture_file", "session_id",
    "timestamp", "icao", "group_id",
]].copy()
assignments["validation_fold"] = -1
for fold_number, (_, val_index) in enumerate(folds, 1):
    assignments.loc[val_index, "validation_fold"] = fold_number
assignments.to_csv(output_dir / "fold_assignments.csv", index=False)

with (output_dir / "label_mapping.json").open("w") as stream:
    json.dump({index: name for index, name in enumerate(class_names)}, stream, indent=2)
serialisable = {key: str(value) if isinstance(value, Path) else value for key, value in CONFIG.items()}
with (output_dir / "experiment_config.json").open("w") as stream:
    json.dump(serialisable, stream, indent=2)
print("Saved:", output_dir.resolve())


Saved: /Users/calebrodgers/Desktop/ml/adsb_specific_transmitter_two_class_outputs


## Notes

Balanced accuracy and macro F1 are used because the class counts are uneven. These are burst-grouped results, not held-out-session results.
